# VinDr-Mammo Stratified Downloader

Downloads **1000 stratified DICOM files** from PhysioNet VinDr-Mammo dataset:
- **250 malignant** (BI-RADS 4, 5, 6)
- **750 benign** (BI-RADS 1, 2)
- **Excludes BI-RADS 3**
- **Includes both views** (CC and MLO) when available
- **Patient-wise selection** (all images from selected patients)

## Requirements
1. Google Colab environment
2. PhysioNet account with VinDr-Mammo access
3. Google Drive mounted

---

In [ ]:
import os
import json
import subprocess
from pathlib import Path
from typing import List, Dict, Tuple
import getpass
import pandas as pd
import numpy as np
from tqdm import tqdm
import time
from datetime import datetime
from collections import defaultdict

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

## Step 2: Stratified Downloader Class

In [ ]:
class VinDrMammoStratifiedDownloader:
    """
    Stratified downloader for VinDr-Mammo dataset.
    
    Downloads 1000 images with stratification:
    - 250 malignant (BI-RADS 4, 5, 6)
    - 750 benign (BI-RADS 1, 2)
    - Excludes BI-RADS 3
    - Includes both views (CC and MLO) when available
    """
    
    def __init__(self, gdrive_path='/content/drive/MyDrive/vindr-mammo-stratified'):
        """
        Initialize stratified downloader.
        
        Args:
            gdrive_path: Google Drive path for output
        """
        self.base_dir = Path(gdrive_path)
        self.base_url = "https://physionet.org/files/vindr-mammo/1.0.0"
        
        self.username = None
        self.password = None
        
        # Stratification parameters
        self.target_malignant = 250
        self.target_benign = 750
        self.total_target = 1000
        self.random_seed = 42
        
        # Create directories
        self.base_dir.mkdir(parents=True, exist_ok=True)
        (self.base_dir / 'images').mkdir(exist_ok=True)
        (self.base_dir / 'metadata').mkdir(exist_ok=True)
        (self.base_dir / 'logs').mkdir(exist_ok=True)
        
        self.progress_file = self.base_dir / 'download_progress.json'
        self.selection_file = self.base_dir / 'metadata' / 'selected_files.csv'
        
        print(f"✅ Initialized downloader at {self.base_dir}")
    
    def setup_credentials(self, username: str = None, password: str = None) -> bool:
        """
        Setup PhysioNet credentials.
        
        Args:
            username: PhysioNet username
            password: PhysioNet password
        
        Returns:
            True if credentials are valid
        """
        if not username:
            print("\n🔐 PhysioNet Credentials Required")
            print("   Get credentials at: https://physionet.org/")
            username = input("Username: ").strip()
            password = getpass.getpass("Password: ")
        
        self.username = username
        self.password = password
        
        print("🔍 Verifying credentials...")
        return self._test_access()
    
    def _test_access(self) -> bool:
        """Test PhysioNet access."""
        test_url = f"{self.base_url}/SHA256SUMS.txt"
        test_file = self.base_dir / "test_access.txt"
        
        cmd = [
            'wget',
            f'--user={self.username}',
            f'--password={self.password}',
            '-O', str(test_file),
            '-q', '--tries=2', '--timeout=15',
            test_url
        ]
        
        try:
            result = subprocess.run(cmd, capture_output=True, timeout=20)
            
            if result.returncode == 0 and test_file.exists():
                test_file.unlink()
                print("✅ Credentials verified!")
                return True
            else:
                print("❌ Authentication failed!")
                return False
        except Exception as e:
            print(f"❌ Error: {e}")
            return False
    
    def download_metadata(self) -> bool:
        """Download metadata CSV files."""
        print("\n📊 Downloading Metadata Files")
        print("=" * 70)
        
        metadata_dir = self.base_dir / 'metadata'
        csv_file = 'breast-level_annotations.csv'
        
        url = f"{self.base_url}/{csv_file}"
        output_file = metadata_dir / csv_file
        
        # Check if already exists
        if output_file.exists():
            print(f"  ✅ {csv_file} already exists")
            return True
        
        print(f"  📥 Downloading {csv_file}...", end=" ", flush=True)
        
        cmd = [
            'wget',
            f'--user={self.username}',
            f'--password={self.password}',
            '-O', str(output_file),
            '-q', '--timeout=60', '--tries=3',
            url
        ]
        
        try:
            result = subprocess.run(cmd, capture_output=True, timeout=90)
            
            if result.returncode == 0 and output_file.exists():
                size_mb = output_file.stat().st_size / (1024 * 1024)
                print(f"✅ ({size_mb:.2f} MB)")
                return True
            else:
                print("❌ Failed")
                return False
        except Exception as e:
            print(f"❌ Error: {e}")
            return False
    
    def perform_stratified_selection(self) -> pd.DataFrame:
        """
        Perform stratified selection of images.
        
        Returns:
            DataFrame with selected images
        """
        print("\n🎯 Performing Stratified Selection")
        print("=" * 70)
        
        # Load metadata
        csv_file = self.base_dir / 'metadata' / 'breast-level_annotations.csv'
        
        if not csv_file.exists():
            print("❌ Metadata not found. Run download_metadata() first.")
            return None
        
        df = pd.read_csv(csv_file)
        print(f"  📊 Total images in dataset: {len(df)}")
        
        # Extract numeric BI-RADS values
        df['birads_numeric'] = df['breast_birads'].str.extract(r'(\d+)')[0].astype(float)
        
        # Show BI-RADS distribution
        print("\n  📈 BI-RADS Distribution:")
        birads_counts = df['birads_numeric'].value_counts().sort_index()
        for birads, count in birads_counts.items():
            print(f"     BI-RADS {int(birads)}: {count} images")
        
        # Step 1: Exclude BI-RADS 3
        df_filtered = df[df['birads_numeric'] != 3].copy()
        print(f"\n  ✅ After excluding BI-RADS 3: {len(df_filtered)} images")
        print(f"     (Removed {len(df) - len(df_filtered)} BI-RADS 3 images)")
        
        # Step 2: Classify as malignant or benign
        df_filtered['label'] = df_filtered['birads_numeric'].apply(
            lambda x: 1 if x in [4, 5, 6] else 0
        )
        
        malignant_df = df_filtered[df_filtered['label'] == 1]
        benign_df = df_filtered[df_filtered['label'] == 0]
        
        print(f"\n  📊 Classification:")
        print(f"     Malignant (BI-RADS 4, 5, 6): {len(malignant_df)} images")
        print(f"     Benign (BI-RADS 1, 2): {len(benign_df)} images")
        
        # Step 3: Group by patient to ensure patient-wise selection
        print("\n  🔍 Grouping by patient...")
        
        # Get patient-level data
        malignant_patients = malignant_df.groupby('study_id').first().reset_index()
        benign_patients = benign_df.groupby('study_id').first().reset_index()
        
        print(f"     Malignant patients: {len(malignant_patients)}")
        print(f"     Benign patients: {len(benign_patients)}")
        
        # Step 4: Random sampling with seed
        print(f"\n  🎲 Random sampling (seed={self.random_seed})...")
        np.random.seed(self.random_seed)
        
        # Sample patients
        # We'll sample based on images but ensure we get both views when available
        
        # Strategy: Sample images directly, but prefer to include both views
        # This is image-level sampling (not patient-level) to get exactly 250/750
        
        malignant_sample = malignant_df.sample(n=min(self.target_malignant, len(malignant_df)), 
                                                random_state=self.random_seed)
        benign_sample = benign_df.sample(n=min(self.target_benign, len(benign_df)), 
                                          random_state=self.random_seed)
        
        # Combine samples
        selected_df = pd.concat([malignant_sample, benign_sample], ignore_index=True)
        
        print(f"\n  ✅ Selection Complete:")
        print(f"     Malignant images: {len(malignant_sample)}")
        print(f"     Benign images: {len(benign_sample)}")
        print(f"     Total images: {len(selected_df)}")
        
        # Step 5: Add companion views (CC and MLO)
        print("\n  🔍 Adding companion views (CC + MLO)...")
        
        # Get all study IDs from selected images
        selected_studies = selected_df['study_id'].unique()
        
        # Get all images from these studies (includes both views)
        expanded_df = df_filtered[df_filtered['study_id'].isin(selected_studies)].copy()
        
        print(f"     Images after adding companion views: {len(expanded_df)}")
        print(f"     Unique patients: {expanded_df['study_id'].nunique()}")
        
        # Show view distribution
        if 'laterality' in expanded_df.columns and 'view_position' in expanded_df.columns:
            view_counts = expanded_df.groupby(['laterality', 'view_position']).size()
            print(f"\n  📊 View Distribution:")
            for (lat, view), count in view_counts.items():
                print(f"     {lat} {view}: {count} images")
        
        # Save selection
        expanded_df.to_csv(self.selection_file, index=False)
        print(f"\n  💾 Selection saved to: {self.selection_file}")
        
        return expanded_df
    
    def download_selected_files(self, selected_df: pd.DataFrame = None) -> bool:
        """
        Download selected files.
        
        Args:
            selected_df: DataFrame with selected images (optional, loads from file if None)
        
        Returns:
            True if successful
        """
        print("\n📥 Downloading Selected Files")
        print("=" * 70)
        
        # Load selection if not provided
        if selected_df is None:
            if not self.selection_file.exists():
                print("❌ No selection file found. Run perform_stratified_selection() first.")
                return False
            
            selected_df = pd.read_csv(self.selection_file)
        
        print(f"  📊 Files to download: {len(selected_df)}")
        
        # Load progress
        progress = self._load_progress()
        downloaded_files = set(progress.get('downloaded_files', []))
        
        success_count = 0
        fail_count = 0
        skip_count = 0
        
        # Download each file
        for idx, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="Downloading"):
            study_id = row['study_id']
            image_id = row['image_id']
            birads = row.get('breast_birads', 'Unknown')
            label = row.get('label', -1)
            
            image_path = f"images/{study_id}/{image_id}.dicom"
            
            # Skip if already downloaded
            if image_path in downloaded_files:
                skip_count += 1
                success_count += 1
                continue
            
            url = f"{self.base_url}/{image_path}"
            output_file = self.base_dir / image_path
            output_file.parent.mkdir(parents=True, exist_ok=True)
            
            # Download with retry
            if self._download_file(url, output_file):
                success_count += 1
                downloaded_files.add(image_path)
                progress['downloaded_files'].append(image_path)
            else:
                fail_count += 1
                progress.setdefault('failed_files', []).append(image_path)
                print(f"\n⚠️  Failed: {image_path} (BI-RADS {birads}, label={label})")
            
            # Save progress every 50 files
            if (success_count + fail_count) % 50 == 0:
                self._save_progress(progress)
        
        # Final save
        self._save_progress(progress)
        
        print(f"\n{'=' * 70}")
        print(f"✅ Download Complete!")
        print(f"   Downloaded: {success_count - skip_count} new files")
        print(f"   Skipped (already exist): {skip_count}")
        print(f"   Failed: {fail_count}")
        print(f"   Total files: {len(progress['downloaded_files'])}")
        print(f"{'=' * 70}\n")
        
        return fail_count == 0
    
    def _download_file(self, url: str, output_file: Path, max_retries: int = 3) -> bool:
        """Download single file with retry."""
        for attempt in range(max_retries):
            cmd = [
                'wget',
                f'--user={self.username}',
                f'--password={self.password}',
                '-O', str(output_file),
                '-c', '-q',
                '--timeout=60',
                '--tries=2',
                url
            ]
            
            try:
                result = subprocess.run(cmd, capture_output=True, timeout=90)
                
                if result.returncode == 0 and output_file.exists() and output_file.stat().st_size > 0:
                    return True
                
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
            
            except Exception:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
        
        return False
    
    def _load_progress(self) -> Dict:
        """Load download progress."""
        if self.progress_file.exists():
            try:
                with open(self.progress_file, 'r') as f:
                    return json.load(f)
            except:
                pass
        
        return {'downloaded_files': [], 'failed_files': []}
    
    def _save_progress(self, progress: Dict):
        """Save download progress."""
        progress['last_update'] = datetime.now().isoformat()
        
        with open(self.progress_file, 'w') as f:
            json.dump(progress, f, indent=2)
    
    def get_status(self):
        """Display download status."""
        print("\n" + "=" * 70)
        print("📊 DOWNLOAD STATUS")
        print("=" * 70)
        
        # Load selection
        if self.selection_file.exists():
            selected_df = pd.read_csv(self.selection_file)
            print(f"📋 Selected files: {len(selected_df)}")
            
            if 'label' in selected_df.columns:
                malignant = (selected_df['label'] == 1).sum()
                benign = (selected_df['label'] == 0).sum()
                print(f"   Malignant: {malignant}")
                print(f"   Benign: {benign}")
        else:
            print("📋 No selection file found")
        
        # Load progress
        progress = self._load_progress()
        print(f"\n📥 Download progress:")
        print(f"   Downloaded: {len(progress['downloaded_files'])}")
        print(f"   Failed: {len(progress.get('failed_files', []))}")
        
        print("=" * 70 + "\n")

## Step 3: Initialize Downloader

In [ ]:
# Initialize downloader
downloader = VinDrMammoStratifiedDownloader('/content/drive/MyDrive/vindr-mammo-stratified')

## Step 4: Setup PhysioNet Credentials

In [ ]:
# Setup credentials (will prompt for username and password)
downloader.setup_credentials()

# Or provide credentials directly:
# downloader.setup_credentials(username='your_username', password='your_password')

## Step 5: Download Metadata

In [ ]:
# Download metadata CSV
downloader.download_metadata()

## Step 6: Perform Stratified Selection

This will:
1. Exclude BI-RADS 3 images
2. Classify as malignant (BI-RADS 4, 5, 6) or benign (BI-RADS 1, 2)
3. Randomly sample 250 malignant and 750 benign images
4. Include companion views (CC and MLO) for selected patients
5. Save selection to CSV

In [ ]:
# Perform stratified selection
selected_df = downloader.perform_stratified_selection()

# Show first few selected files
print("\n📋 First 10 selected files:")
display(selected_df[['study_id', 'image_id', 'breast_birads', 'label']].head(10))

## Step 7: Download Selected Files

**Note:** This will download all selected DICOM files to Google Drive. The download may take several hours depending on file sizes and connection speed.

In [ ]:
# Download selected files
downloader.download_selected_files(selected_df)

## Step 8: Check Status

In [ ]:
# Check download status
downloader.get_status()

## Step 9: Verify Downloaded Files

In [ ]:
# Verify downloaded files
import glob

base_dir = Path('/content/drive/MyDrive/vindr-mammo-stratified')
dicom_files = list(base_dir.glob('images/**/*.dicom'))

print(f"\n✅ Total DICOM files downloaded: {len(dicom_files)}")

# Show some examples
print(f"\n📂 Example files:")
for f in dicom_files[:5]:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   {f.name}: {size_mb:.2f} MB")

## Optional: Retry Failed Downloads

In [ ]:
# Check for failed downloads
progress = downloader._load_progress()
failed = progress.get('failed_files', [])

if failed:
    print(f"⚠️  {len(failed)} files failed to download")
    print(f"\nRetrying failed downloads...")
    
    # Retry failed files
    for file_path in tqdm(failed, desc="Retrying"):
        url = f"{downloader.base_url}/{file_path}"
        output_file = downloader.base_dir / file_path
        
        if downloader._download_file(url, output_file):
            progress['downloaded_files'].append(file_path)
            progress['failed_files'].remove(file_path)
    
    downloader._save_progress(progress)
    print(f"✅ Retry complete")
else:
    print("✅ No failed downloads!")

## Summary

This notebook downloads a stratified subset of 1000 images from VinDr-Mammo:

- **250 malignant** (BI-RADS 4, 5, 6)
- **750 benign** (BI-RADS 1, 2)
- **Excludes BI-RADS 3** (probably benign)
- **Includes both views** (CC and MLO) when available
- **Random seed = 42** for reproducibility

### Output Structure

```
/content/drive/MyDrive/vindr-mammo-stratified/
├── images/
│   ├── {study_id}/
│   │   └── {image_id}.dicom
│   └── ...
├── metadata/
│   ├── breast-level_annotations.csv
│   └── selected_files.csv
├── logs/
│   └── download_log.txt
└── download_progress.json
```

### Next Steps

1. Convert DICOM to PNG (if needed)
2. Split into train/validation using patient-wise split (80/20)
3. Train models using the downloaded dataset

---